In [1]:
import pandas as pd
import rdkit.Chem as Chem

from config import SplitType, TrainConfig
from misc import enable_determinism
from models import chemprop_modded_ref as cpm_ref
from models import chemprop_ref as cp_ref
from preprocessing import get_butina_clusters, mol_to_inchi, standardize
from train import generate_kfold_splits, train_and_evaluate

enable_determinism()


In [2]:
df = pd.read_csv("./datasets/ADME_public_set_3521.csv")
df = df.loc[:, ["SMILES", "LOG HLM_CLint (mL/min/kg)"]]
df.columns = ["smiles", "target"]
# df = df.sample(2000)
df = df.dropna(subset="target").reset_index(drop=True)

df["mol"] = df["smiles"].map(standardize)
df = df.dropna(subset='mol').reset_index(drop=True)
df["inchi"] = df["mol"].map(mol_to_inchi)
df["mol"] = df["inchi"].map(Chem.MolFromInchi)
df["butina_cluster"] = get_butina_clusters(df["mol"])
len(df)
# df

3087

In [3]:
def evaluate(df, cp_config, cpm_config=None):
    if cpm_config is None:
            cpm_config = cp_config

    splits = generate_kfold_splits(
        df,
        split_type=SplitType.BUTINA,
        n_folds=3,
        random_state=cp_config.random_state,
    )

    results = []
    for fold, (train_df, val_df, test_df) in enumerate(splits):
        cp_results_dict = cp_ref.train_and_evaluate_on_split(train_df, val_df, test_df, cp_config)
        cp_results_dict['model'] = 'baseline' # type: ignore

        cpm_results_dict = cpm_ref.train_and_evaluate_on_split(train_df, val_df, test_df, cpm_config)
        cpm_results_dict['model'] = 'modded' # type: ignore

        result_dict = {**cp_results_dict, **cpm_results_dict}
        result_dict['fold'] = fold
        results.append(result_dict)

    return pd.DataFrame.from_records(results)

In [4]:
cfg = TrainConfig(max_epochs=20)
results_df = evaluate(df, cfg)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/rahul/chemprop_mods/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ NormAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ Identity           │      0 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 26                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 0.911


...

Metric val_loss improved by 0.010 >= min_delta = 0.0. New best score: 0.680


Metric val_loss improved by 0.003 >= min_delta = 0.0. New best score: 0.677
`Trainer.fit` stopped: `max_epochs=20` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/mae          │    0.46665656566619873    │
│          test/r2          │    0.2084447145462036     │
│         test/rmse         │    0.5623066425323486     │
└───────────────────────────┴───────────────────────────┘

In [5]:
results_df.to_csv("./fin.csv")